In [1]:
from transformers import BertModel, BertTokenizer
import torch

In [2]:
from progressbar import *
from tqdm import tqdm

In [3]:
use_cuda = torch.cuda.is_available()
dataset="new2"

In [4]:
device = torch.device('cuda' if use_cuda else 'cpu')
print(device)

cuda


In [5]:
#测试
sentences = [
    "This is the first sentence.",
    "This is the second sentence.",
    "Here is another sentence.",
    # 添加更多的句子...
]

In [6]:
#加速器
import subprocess
import os

result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
output = result.stdout
for line in output.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value

In [7]:
#创建一个进度条
def get_bert_embedding(sentences, similarity_threshold = 0.5):
  pbar = ProgressBar(widgets=['Getting wordwise similarity ', Percentage(), ' ', Bar(),
        ' ', ETA(), ' '], maxval=len(sentences)).start()
  total_set = {}
  valid_set={}
#分词器
  #tokenizer = BertTokenizer.from_pretrained("D:/ProgramData/bert-base-uncased")
  tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

  #model = BertModel.from_pretrained("D:/ProgramData/bert-base-uncased").to(device)
  model = BertModel.from_pretrained("bert-base-uncased").to(device)
  count = 0
  for i in tqdm(range(0,len(sentences))):
    # print(i)
    s=sentences[i]
  # for s in sentences:
    pbar.update(count)
    encoded_input = tokenizer(s, return_tensors='pt').to(device)
    output = model(**encoded_input)
    aa = torch.squeeze(output.last_hidden_state,dim=0)
    each = aa[1:-1,:]
    split_s = s.split()
    for word in range(0,len(split_s)):
      for another in range(0,len(split_s)):
        w1 = split_s[word]
        w2 = split_s[another]
        score = torch.cosine_similarity(each[word].view(1,-1),each[another].view(1,-1),dim=1)#计算相似度(同一个句子不同单词之间的相似度)
        key = ''+w1+','+w2
        if key not in total_set.keys():
          total_set[key] = 1
        else:
          total_set[key]+=1
        if score.data>similarity_threshold:
          if key not in valid_set.keys():
            valid_set[key] = 1
          else:
            valid_set[key] += 1
    count +=1
  pbar.finish()
  return total_set,valid_set


In [8]:
# DO NOT TOKENIZE YOUR SENTENCE!!!!!! [[sentence0],[sentence1],[sentence2],[sentence3],.......]
sentences=[]
file=open(f'../../data_tgcn/{dataset}/{dataset}_new.clean.txt', "r", encoding="utf-8").readlines()
for line in file:
  sentences.append(line[:-1])
total_set,valid_set = get_bert_embedding(sentences)
# print(total_set)
# print(valid_set)
import pickle

100%|██████████| 19263/19263 [11:48<00:00, 27.17it/s]         | ETA:  --:--:-- Getting wordwise similarity   0% |                            | ETA:  --:--:-- Getting wordwise similarity   0% |                            | ETA:  14:49:21 Getting wordwise similarity   0% |                             | ETA:  0:35:33 Getting wordwise similarity   0% |                             | ETA:  0:22:29 Getting wordwise similarity   0% |                             | ETA:  0:18:06 Getting wordwise similarity   0% |                             | ETA:  0:15:24 Getting wordwise similarity   0% |                             | ETA:  0:15:49 Getting wordwise similarity   0% |                             | ETA:  0:15:14 Getting wordwise similarity   1% |                             | ETA:  0:13:58 Getting wordwise similarity   1% |                             | ETA:  0:13:38 Getting wordwise similarity   1% |                             | ETA:  0:13:42 Getting wordwise similarity   1% |                  

In [9]:
save_path=f'../../data_tgcn/{dataset}/bert'
if not os.path.exists(save_path):
  os.mkdir(save_path)
f = open(os.path.join(save_path,'{dataset}_semantic.pkl'), 'wb')
pickle.dump(valid_set,f)
f.close()